<a href="https://colab.research.google.com/github/Psynotch/firstrep/blob/master/Session10_Extended_Turing_Verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip -q install python-flint==0.9.0

In [2]:
from flint import acb, arb, arb_series, ctx

from fractions import Fraction
import csv
import json
import time

ctx.prec = 256

T_LO = arb("10")
T_MID = arb("1000")
T_HI = arb("5000")

print("SESSION 10 INITIALIZED")
print("Precision:", ctx.prec, "bits")
print("Height range: 10 to 5000")

SESSION 10 INITIALIZED
Precision: 256 bits
Height range: 10 to 5000


In [3]:
#Obtain rigorous zero counts
print("SESSION 10: TURING ZERO COUNTS")
print("------------------------------")

start = time.time()

N10_ball = T_LO.zeta_nzeros()
N1000_ball = T_MID.zeta_nzeros()
N5000_ball = T_HI.zeta_nzeros()

N10 = N10_ball.unique_fmpz()
N1000 = N1000_ball.unique_fmpz()
N5000 = N5000_ball.unique_fmpz()

if any(x is None for x in [N10, N1000, N5000]):
    raise RuntimeError("One or more counts do not isolate a unique integer.")

N10 = int(N10)
N1000 = int(N1000)
N5000 = int(N5000)

count_10_1000 = N1000 - N10
count_1000_5000 = N5000 - N1000
total_10_5000 = N5000 - N10

print("N(10):", N10)
print("N(1000):", N1000)
print("N(5000):", N5000)

print("\nZeros from 10 to 1000:", count_10_1000)
print("Zeros from 1000 to 5000:", count_1000_5000)
print("TOTAL FROM 10 TO 5000:", total_10_5000)

print("\nMatches Session 08 count:",
      count_10_1000 == 649)

print("Elapsed seconds:", round(time.time() - start, 3))

assert count_10_1000 == 649
assert count_1000_5000 > 0

SESSION 10: TURING ZERO COUNTS
------------------------------
N(10): 0
N(1000): 649
N(5000): 4520

Zeros from 10 to 1000: 649
Zeros from 1000 to 5000: 3871
TOTAL FROM 10 TO 5000: 4520

Matches Session 08 count: True
Elapsed seconds: 0.004


In [4]:
#Generate all certified zero enclosures
print("GENERATING ZERO ENCLOSURES")
print("---------------------------")

start = time.time()

first_index = N10 + 1

zeros = list(
    acb.zeta_zeros(first_index, total_10_5000)
)

print("Enclosures generated:", len(zeros))
print("Expected:", total_10_5000)

assert len(zeros) == total_10_5000

print("\nFIRST THREE")

for zero in zeros[:3]:
    print(zero)

print("\nLAST THREE")

for zero in zeros[-3:]:
    print(zero)

print("\nElapsed seconds:", round(time.time() - start, 3))

GENERATING ZERO ENCLOSURES
---------------------------
Enclosures generated: 4520
Expected: 4520

FIRST THREE
0.5000000000000000000000000000000000000000000000000000000000000000000000000000 + [14.13472514173469379045725198356247027078425711569924317568556746014996342981 +/- 8.84e-76]j
0.5000000000000000000000000000000000000000000000000000000000000000000000000000 + [21.02203963877155499262847959389690277733434052490278175462952040358759858607 +/- 1.61e-75]j
0.5000000000000000000000000000000000000000000000000000000000000000000000000000 + [25.01085758014568876321379099256282181865954967255799667249654200674509209844 +/- 1.80e-75]j

LAST THREE
0.5000000000000000000000000000000000000000000000000000000000000000000000000000 + [4997.396150707099834388128964987988480763367788704569557114978192444847845197 +/- 2.52e-73]j
0.5000000000000000000000000000000000000000000000000000000000000000000000000000 + [4998.370030195671724560514313424394594765937791674344664733703696750472578800 +/- 2.28e-73]j
0.5

In [5]:
#Construct adaptive rational intervals
def rational_ball(q):
    return arb(f"{q.numerator}/{q.denominator}")


centers = [
    Fraction(z.imag.mid().str(60, radius=False))
    for z in zeros
]

records = []

for i, zero in enumerate(zeros):

    center = centers[i]
    gamma = zero.imag

    radius = Fraction(1, 100)

    # Keep enough separation from the previous zero.
    if i > 0:
        radius = min(
            radius,
            (center - centers[i - 1]) / 4
        )

    # Keep enough separation from the next zero.
    if i + 1 < len(centers):
        radius = min(
            radius,
            (centers[i + 1] - center) / 4
        )

    # Stay inside the requested height range.
    radius = min(
        radius,
        (center - Fraction(10)) / 4,
        (Fraction(5000) - center) / 4
    )

    if radius <= 0:
        raise RuntimeError(f"Invalid interval for zero {i+1}")

    left = center - radius
    right = center + radius

    # Rigorous enclosure-containment test.
    left_ball = rational_ball(left)
    right_ball = rational_ball(right)

    contained = (
        left_ball.upper() < gamma.lower()
        and gamma.upper() < right_ball.lower()
    )

    if not contained:
        raise RuntimeError(
            f"Zero {i+1}: rational interval does not contain its enclosure."
        )

    records.append({
        "index": first_index + i,
        "zero": zero,
        "gamma": gamma,
        "left": left,
        "right": right,
        "radius": radius
    })


# Check that the resulting intervals do not overlap.
for i in range(len(records) - 1):
    if not records[i]["right"] < records[i + 1]["left"]:
        raise RuntimeError(f"Intervals {i+1} and {i+2} overlap.")


print("Intervals created:", len(records))
print("All zero enclosures contained: True")
print("All intervals disjoint: True")
print("All intervals inside (10,5000): True")

Intervals created: 4520
All zero enclosures contained: True
All intervals disjoint: True
All intervals inside (10,5000): True


In [6]:
#Verify Hardy \(Z\)-function sign changes
def hardy_z(t_fraction):

    t = rational_ball(t_fraction)

    result = arb_series(
        [t, 1],
        prec=2
    ).riemann_siegel_z()

    return result.coeffs()[0]


def certified_sign(value):

    sign = value.sgn()

    if sign == arb(1):
        return "+"

    if sign == arb(-1):
        return "-"

    return "UNKNOWN"


verification_rows = []
verified = 0
failed = []

print("SESSION 10: HARDY Z SIGN VERIFICATION")
print("-------------------------------------")

start = time.time()

for i, record in enumerate(records, start=1):

    z_left = hardy_z(record["left"])
    z_right = hardy_z(record["right"])

    sign_left = certified_sign(z_left)
    sign_right = certified_sign(z_right)

    passed = (
        sign_left != "UNKNOWN"
        and sign_right != "UNKNOWN"
        and sign_left != sign_right
    )

    if passed:
        verified += 1
    else:
        failed.append(record["index"])

    verification_rows.append({
        "zero_index": record["index"],
        "left": str(record["left"]),
        "right": str(record["right"]),
        "left_sign": sign_left,
        "right_sign": sign_right,
        "passed": passed
    })

    if i % 250 == 0 or i == len(records):
        print(
            f"Checked {i}/{len(records)} | "
            f"Passed: {verified} | "
            f"Failed: {len(failed)}"
        )

print("\nVERIFIED:", verified)
print("FAILED:", len(failed))
print("Failed zero indices:", failed)

print("Elapsed seconds:", round(time.time() - start, 3))

SESSION 10: HARDY Z SIGN VERIFICATION
-------------------------------------
Checked 250/4520 | Passed: 250 | Failed: 0
Checked 500/4520 | Passed: 500 | Failed: 0
Checked 750/4520 | Passed: 750 | Failed: 0
Checked 1000/4520 | Passed: 1000 | Failed: 0
Checked 1250/4520 | Passed: 1250 | Failed: 0
Checked 1500/4520 | Passed: 1500 | Failed: 0
Checked 1750/4520 | Passed: 1750 | Failed: 0
Checked 2000/4520 | Passed: 2000 | Failed: 0
Checked 2250/4520 | Passed: 2250 | Failed: 0
Checked 2500/4520 | Passed: 2500 | Failed: 0
Checked 2750/4520 | Passed: 2750 | Failed: 0
Checked 3000/4520 | Passed: 3000 | Failed: 0
Checked 3250/4520 | Passed: 3250 | Failed: 0
Checked 3500/4520 | Passed: 3500 | Failed: 0
Checked 3750/4520 | Passed: 3750 | Failed: 0
Checked 4000/4520 | Passed: 4000 | Failed: 0
Checked 4250/4520 | Passed: 4250 | Failed: 0
Checked 4500/4520 | Passed: 4500 | Failed: 0
Checked 4520/4520 | Passed: 4520 | Failed: 0

VERIFIED: 4520
FAILED: 0
Failed zero indices: []
Elapsed seconds: 25.387


In [7]:
with open(
    "Session10_full_zero_certificate.csv",
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "zero_index",
            "complex_zero_enclosure",
            "ordinate_enclosure",
            "left",
            "right",
            "left_sign",
            "right_sign",
            "passed"
        ]
    )

    writer.writeheader()

    for record, check in zip(records, verification_rows):

        writer.writerow({
            "zero_index": record["index"],
            "complex_zero_enclosure": str(record["zero"]),
            "ordinate_enclosure": str(record["gamma"]),
            "left": str(record["left"]),
            "right": str(record["right"]),
            "left_sign": check["left_sign"],
            "right_sign": check["right_sign"],
            "passed": check["passed"]
        })


final_success = (
    count_10_1000 == 649
    and total_10_5000 == len(zeros)
    and verified == len(records)
    and len(failed) == 0
)

summary = {
    "session": 10,
    "height_range": [10, 5000],
    "precision_bits": 256,
    "N_10": N10,
    "N_1000": N1000,
    "N_5000": N5000,
    "previous_range_count": count_10_1000,
    "new_range_count": count_1000_5000,
    "total_range_count": total_10_5000,
    "isolated_zero_enclosures": len(zeros),
    "hardy_sign_changes": verified,
    "failed_intervals": failed,
    "success": final_success
}

with open(
    "Session10_verification_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2)


print("SESSION 10 FINAL REPORT")
print("=======================")
print(json.dumps(summary, indent=2))

if final_success:
    print("\nSESSION 10: SUCCESS")
else:
    print("\nSESSION 10: NOT YET COMPLETE")

SESSION 10 FINAL REPORT
{
  "session": 10,
  "height_range": [
    10,
    5000
  ],
  "precision_bits": 256,
  "N_10": 0,
  "N_1000": 649,
  "N_5000": 4520,
  "previous_range_count": 649,
  "new_range_count": 3871,
  "total_range_count": 4520,
  "isolated_zero_enclosures": 4520,
  "hardy_sign_changes": 4520,
  "failed_intervals": [],
  "success": true
}

SESSION 10: SUCCESS
